# Power analysis

Pre-registered: for H1's day-level test, the smallest true difference in mean arrival delay detectable with 80% power at alpha = 0.05 given the observed SD and number of days; and for the flight-level test, the same with tens of thousands of flights, to show why statistical significance alone means little here.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from delaystats import plotting, results
from delaystats import stats_helpers as sh
from delaystats.athena import run_query

ALPHA = 0.05  # pre-registered, two-sided (docs/ANALYSIS_PLAN.md)
SEED = 20261014  # fixed for every bootstrap
pd.set_option("display.float_format", "{:,.4f}".format)


def fmt_p(p):
    return "< 0.0001" if p < 1e-4 else f"{p:.4f}"


def table(rs):
    return pd.DataFrame([r.as_row() for r in rs]).set_index("test")


from statsmodels.stats.power import TTestIndPower

In [ ]:
flights = run_query("h1_sea_summer_flights")
daily = flights.groupby(["year", "flightdate"], as_index=False)["arrdelay"].mean()


def pooled_sd(x, y):
    return float(
        np.sqrt(
            ((len(x) - 1) * x.var(ddof=1) + (len(y) - 1) * y.var(ddof=1)) / (len(x) + len(y) - 2)
        )
    )


day_a = daily.loc[daily["year"] == 2024, "arrdelay"].to_numpy(float)
day_b = daily.loc[daily["year"] == 2023, "arrdelay"].to_numpy(float)
fl_a = flights.loc[flights["year"] == 2024, "arrdelay"].to_numpy(float)
fl_b = flights.loc[flights["year"] == 2023, "arrdelay"].to_numpy(float)

day = sh.power_min_detectable(
    pooled_sd(day_a, day_b), len(day_a), len(day_b), alpha=ALPHA, name="power_h1_day_level"
)
flight = sh.power_min_detectable(
    pooled_sd(fl_a, fl_b), len(fl_a), len(fl_b), alpha=ALPHA, name="power_h1_flight_level"
)
results.save("POWER", [day, flight])
table([day, flight])

In [ ]:
sd_day = pooled_sd(day_a, day_b)
diffs = np.linspace(0.1, 15, 150)
power = TTestIndPower().power(
    effect_size=diffs / sd_day, nobs1=len(day_a), ratio=len(day_b) / len(day_a), alpha=ALPHA
)
plotting.power_curve(
    diffs,
    power,
    day.estimate,
    "power_h1_day_level",
    f"Power of the H1 day-level test ({len(day_a)} vs {len(day_b)} days)",
)

## Interpretation

In [ ]:
display(
    Markdown(f"""
- **Day level (the honest test for H1):** with {len(day_a)} and {len(day_b)} days and a pooled SD of daily
  means of {pooled_sd(day_a, day_b):.2f} minutes, the smallest true difference detectable with 80% power at
  alpha = {ALPHA} is **{day.estimate:.2f} minutes** (Cohen's d = {day.effect_size:.3f}). A real change smaller
  than that would usually go undetected, so a non-significant day-level result is not evidence of no change.
- **Flight level:** with {len(fl_a):,} and {len(fl_b):,} flights, the same calculation gives
  **{flight.estimate:.2f} minutes** (d = {flight.effect_size:.4f}). Even a half-minute difference, which no operations
  leader would act on, is detectable. This is why every finding is judged against the pre-registered
  2-minute practical threshold, not against p alone.
""")
)